In [19]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, roc_auc_score
import torch.nn.functional as F


### Loading model

In [20]:
class CnnBetter(nn.Module):
    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3):
        super().__init__()
        def block(in_c, out_c, drop):
            # Podwajamy kanały gdy zmniejszamy mapę o połowę — zachowujemy
            # całkowitą "pojemność informacyjną" warstwy (in_c * H * W ≈ out_c * H/2 * W/2)
            return [
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(drop),
            ]
        self.layers = nn.Sequential(
            *block(3,  32,  dropout_p),   # kanały: 3→32,   mapa: 32×32→16×16
            *block(32, 64,  dropout_p),   # kanały: 32→64,  mapa: 16×16→8×8
            *block(64, 128, dropout_p),   # kanały: 64→128, mapa: 8×8→4×4
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):         return self.layers(x)
    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)
    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)

In [21]:
model = torch.load('CNN_ROBUST_cifar10.pth', weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CnnBetter(num_classes=10)
state_dict = torch.load('CNN_ROBUST_cifar10.pth', map_location=device)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

CnnBetter(
  (layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.3, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.3, inplace=F

### Creating set of labels produced by victim model

In [32]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,          # pełen zbiór treningowy
    download=False,
    transform=transform
)

loader = DataLoader(dataset, batch_size=128, shuffle=False)

all_preds = [] # hard labels - numbers from 0-9 (my quasi-labels)
all_probs = [] #

model.eval()

with torch.no_grad():
    for x, _ in loader:
        x = x.to(device)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())


all_preds = torch.cat(all_preds).numpy()
all_probs = torch.cat(all_probs).numpy()

np.save('victim_preds.npy', all_preds)
np.save('victim_probs.npy', all_probs)

/Users/jakubwoszczek-fullfocus/Kubiszon/Studia/Sem6/ai_safety/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


#### Validation of loaded model

In [27]:
from sklearn.metrics import accuracy_score

true_labels = np.array(dataset.targets)
print("Accuracy:", accuracy_score(true_labels, all_preds))

Accuracy: 0.8428


### Training surrogate on quasi-labels

In [28]:
class PseudoDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, pseudo_labels):
        self.dataset = original_dataset
        self.labels = pseudo_labels

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]  # ignorujemy prawdziwą etykietę
        y = self.labels[idx]
        return x, y

class CnnSurrogate(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [37]:
pseudo_dataset = PseudoDataset(dataset, all_preds)
eval_loader = DataLoader(pseudo_dataset, batch_size=128, shuffle=False)

Training

In [33]:
class PseudoDatasetSoft(torch.utils.data.Dataset):
    def __init__(self, original_dataset, soft_labels):
        self.dataset = original_dataset
        self.labels = soft_labels

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        y = self.labels[idx]
        return x, torch.tensor(y, dtype=torch.float32)

In [35]:
pseudo_dataset = PseudoDatasetSoft(dataset, all_probs)
loader = DataLoader(pseudo_dataset, batch_size=128, shuffle=True)

model_surr = CnnSurrogate().to(device)
optimizer = torch.optim.Adam(model_surr.parameters(), lr=1e-3)

criterion = nn.KLDivLoss(reduction="batchmean")

epochs = 5

for epoch in range(epochs):
    model_surr.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model_surr(x)
        log_probs = F.log_softmax(logits, dim=1)

        loss = criterion(log_probs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    print(f"Epoch {epoch+1}, loss: {total_loss:.3f}")

Epoch 1, loss: 219.912
Epoch 2, loss: 105.431
Epoch 3, loss: 72.012
Epoch 4, loss: 54.787
Epoch 5, loss: 43.797


surrogate validation : zgodność predykcji surrogate z predykcjami victim

In [38]:
model_surr.eval()
preds = []

with torch.no_grad():
    for x, _ in eval_loader:
        x = x.to(device)
        out = model_surr(x)
        preds.append(torch.argmax(out, dim=1).cpu())

preds = torch.cat(preds).numpy()

print("Agreement:", accuracy_score(all_preds, preds))

Agreement: 0.87482


### atak adwersarialny metodą PGD lub FGSM na surrogate i oceń skuteczność ataku na victim

In [39]:
def fgsm_attack(model, x, y, epsilon=0.03):
    x_adv = x.clone().detach().requires_grad_(True)

    outputs = model(x_adv)
    loss = F.cross_entropy(outputs, y)

    model.zero_grad()
    loss.backward()

    x_adv = x_adv + epsilon * x_adv.grad.sign()
    x_adv = torch.clamp(x_adv, 0, 1)

    return x_adv.detach()

In [40]:
model_surr.eval()

adv_samples = []
true_labels = []

for x, y in eval_loader:
    x = x.to(device)
    y = y.to(device)

    x_adv = fgsm_attack(model_surr, x, y, epsilon=0.03)

    adv_samples.append(x_adv.cpu())
    true_labels.append(y.cpu())

adv_samples = torch.cat(adv_samples)
true_labels = torch.cat(true_labels)

In [41]:
model.eval()

victim_preds = []

with torch.no_grad():
    for i in range(0, len(adv_samples), 128):
        x = adv_samples[i:i+128].to(device)

        out = model(x)
        preds = torch.argmax(out, dim=1)

        victim_preds.append(preds.cpu())

victim_preds = torch.cat(victim_preds)

In [42]:
orig_acc = accuracy_score(true_labels.numpy(), all_preds)  # baseline
adv_acc = accuracy_score(true_labels.numpy(), victim_preds.numpy())

print("Victim accuracy on clean:", orig_acc)
print("Victim accuracy on adversarial:", adv_acc)
print("Attack success rate:", 1 - adv_acc)

Victim accuracy on clean: 1.0
Victim accuracy on adversarial: 0.54856
Attack success rate: 0.45143999999999995


### proces budowy surrogate, ograniczając się do maksymalnie 1000 zapytań do victim

In [43]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import accuracy_score
import torch.nn.functional as F


### Loading victim model
class CnnBetter(nn.Module):
    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3):
        super().__init__()

        def block(in_c, out_c, drop):
            return [
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(drop),
            ]

        self.layers = nn.Sequential(
            *block(3, 32, dropout_p),
            *block(32, 64, dropout_p),
            *block(64, 128, dropout_p),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):         return self.layers(x)

    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)

    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = CnnBetter(num_classes=10)
state_dict = torch.load('CNN_ROBUST_cifar10.pth', map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

### LIMITED QUERIES: Select only 1000 samples from CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

full_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# OGRANICZENIE: wybieramy losowo tylko 1000 próbek
MAX_QUERIES = 1000
np.random.seed(42)
selected_indices = np.random.choice(len(full_dataset), size=MAX_QUERIES, replace=False)

# Tworzymy subset
limited_dataset = Subset(full_dataset, selected_indices)
print(f"Limited dataset size: {len(limited_dataset)} samples (max queries)")

### Query victim model on limited dataset (1000 queries)
loader = DataLoader(limited_dataset, batch_size=128, shuffle=False)

all_preds = []
all_probs = []

model.eval()
query_count = 0

with torch.no_grad():
    for x, _ in loader:
        x = x.to(device)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())

        query_count += len(x)

print(f"Total queries made to victim: {query_count}")

all_preds = torch.cat(all_preds).numpy()
all_probs = torch.cat(all_probs).numpy()

np.save('victim_preds_limited.npy', all_preds)
np.save('victim_probs_limited.npy', all_probs)


### Define surrogate model
class CnnSurrogate(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


### Prepare soft-label dataset for training
class PseudoDatasetSoft(torch.utils.data.Dataset):
    def __init__(self, original_dataset, soft_labels):
        self.dataset = original_dataset
        self.labels = soft_labels

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        y = self.labels[idx]
        return x, torch.tensor(y, dtype=torch.float32)


pseudo_dataset = PseudoDatasetSoft(limited_dataset, all_probs)
train_loader = DataLoader(pseudo_dataset, batch_size=64, shuffle=True)

### Train surrogate model on soft labels from victim
model_surr = CnnSurrogate().to(device)
optimizer = torch.optim.Adam(model_surr.parameters(), lr=1e-3)
criterion = nn.KLDivLoss(reduction="batchmean")

epochs = 20  # Więcej epok bo mamy mniej danych

print("Training surrogate model...")
for epoch in range(epochs):
    model_surr.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model_surr(x)
        log_probs = F.log_softmax(logits, dim=1)
        loss = criterion(log_probs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, loss: {total_loss:.3f}")

torch.save(model_surr.state_dict(), 'surrogate_limited.pth')
print("Surrogate model saved!")

### Validate surrogate: agreement with victim on training set
eval_loader = DataLoader(limited_dataset, batch_size=128, shuffle=False)

model_surr.eval()
surr_preds = []

with torch.no_grad():
    for x, _ in eval_loader:
        x = x.to(device)
        out = model_surr(x)
        surr_preds.append(torch.argmax(out, dim=1).cpu())

surr_preds = torch.cat(surr_preds).numpy()

agreement = accuracy_score(all_preds, surr_preds)
print(f"\nSurrogate-Victim Agreement on {MAX_QUERIES} samples: {agreement:.4f}")


### Test transferability: FGSM attack on surrogate → test on victim
def fgsm_attack(model, x, y, epsilon=0.03):
    x_adv = x.clone().detach().requires_grad_(True)
    outputs = model(x_adv)
    loss = F.cross_entropy(outputs, y)

    model.zero_grad()
    loss.backward()

    x_adv = x_adv + epsilon * x_adv.grad.sign()
    x_adv = torch.clamp(x_adv, 0, 1)

    return x_adv.detach()


print("\nGenerating adversarial examples using FGSM on surrogate...")
model_surr.eval()

adv_samples = []
labels = []

for x, y in eval_loader:
    x = x.to(device)
    y = y.to(device)

    x_adv = fgsm_attack(model_surr, x, y, epsilon=0.03)

    adv_samples.append(x_adv.cpu())
    labels.append(y.cpu())

adv_samples = torch.cat(adv_samples)
labels = torch.cat(labels)

### Evaluate attack success on victim model
print("Testing adversarial examples on victim model...")
model.eval()

victim_preds_clean = []
victim_preds_adv = []

with torch.no_grad():
    for i in range(0, len(adv_samples), 128):
        # Clean samples
        x_clean = []
        for idx in range(i, min(i + 128, len(limited_dataset))):
            img, _ = limited_dataset[idx]
            x_clean.append(img)
        x_clean = torch.stack(x_clean).to(device)

        out_clean = model(x_clean)
        victim_preds_clean.append(torch.argmax(out_clean, dim=1).cpu())

        # Adversarial samples
        x_adv = adv_samples[i:i + 128].to(device)
        out_adv = model(x_adv)
        victim_preds_adv.append(torch.argmax(out_adv, dim=1).cpu())

victim_preds_clean = torch.cat(victim_preds_clean)
victim_preds_adv = torch.cat(victim_preds_adv)

clean_acc = accuracy_score(labels.numpy(), victim_preds_clean.numpy())
adv_acc = accuracy_score(labels.numpy(), victim_preds_adv.numpy())
attack_success = 1 - adv_acc

print("\n" + "=" * 60)
print("RESULTS WITH LIMITED QUERIES")
print("=" * 60)
print(f"Number of queries to victim: {MAX_QUERIES}")
print(f"Surrogate-Victim agreement: {agreement:.4f}")
print(f"\nVictim accuracy on clean samples: {clean_acc:.4f}")
print(f"Victim accuracy on adversarial samples: {adv_acc:.4f}")
print(f"Attack Success Rate: {attack_success:.4f}")
print("=" * 60)

Using device: cpu


/Users/jakubwoszczek-fullfocus/Kubiszon/Studia/Sem6/ai_safety/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Limited dataset size: 1000 samples (max queries)
Total queries made to victim: 1000
Training surrogate model...
Epoch 5/20, loss: 9.931
Epoch 10/20, loss: 4.871
Epoch 15/20, loss: 1.903
Epoch 20/20, loss: 0.981
Surrogate model saved!

Surrogate-Victim Agreement on 1000 samples: 0.9580

Generating adversarial examples using FGSM on surrogate...
Testing adversarial examples on victim model...

RESULTS WITH LIMITED QUERIES
Number of queries to victim: 1000
Surrogate-Victim agreement: 0.9580

Victim accuracy on clean samples: 0.8600
Victim accuracy on adversarial samples: 0.5270
Attack Success Rate: 0.4730


### powtórz proces budowy surrogate, ograniczając się do maksymalnie 1000 zapytań do victim